In [27]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("seyyedaliayati/solidity-dataset")

In [30]:
for split, dataset in ds.items():
    print(f"Features in {split} set: {dataset.features}")

Features in train set: {'hash': Value(dtype='string', id=None), 'size': Value(dtype='int64', id=None), 'ext': Value(dtype='string', id=None), 'lang': Value(dtype='string', id=None), 'is_test': Value(dtype='bool', id=None), 'repo_id': Value(dtype='string', id=None), 'repo_name': Value(dtype='string', id=None), 'repo_head': Value(dtype='string', id=None), 'repo_path': Value(dtype='string', id=None), 'content_tokens': Value(dtype='int64', id=None), 'content_chars': Value(dtype='int64', id=None), 'content': Value(dtype='string', id=None), '__index_level_0__': Value(dtype='int64', id=None)}
Features in test set: {'hash': Value(dtype='string', id=None), 'size': Value(dtype='int64', id=None), 'ext': Value(dtype='string', id=None), 'lang': Value(dtype='string', id=None), 'is_test': Value(dtype='bool', id=None), 'repo_id': Value(dtype='string', id=None), 'repo_name': Value(dtype='string', id=None), 'repo_head': Value(dtype='string', id=None), 'repo_path': Value(dtype='string', id=None), 'conten

The following function is to access the readme files of the following repositories in our dataset.

In [32]:
ds['train'][0]['repo_name']

'JolyonJian/contracts'

In [40]:
import requests
from datasets import DatasetDict
import base64
import pandas as pd
from typing import Optional
import logging



class TransformedDataset:
    def __init__(self, gh_token, dataset: DatasetDict, num_train_examples: int = 300):
        self.gh_token = gh_token
        assert(num_train_examples >= 10)
        self.train_set = dataset['train'].select(range(num_train_examples))
        self.test_set = dataset['test'].select(range(int(num_train_examples * 0.1)))
        self.eval_set = dataset['eval'].select(range(int(num_train_examples * 0.1)))
        self.create_readme_dataset(self.gh_token)


    def fetch_readme_content(self, repo_name: str) -> Optional[str]:
        """
        Fetch README.md content from a GitHub repository.
        
        Args:
            repo_name (str): Repository name
            token (str, optional): GitHub personal access token for authentication
        
        Returns:
            str: README content if found, None otherwise
        """
        # Setup logging
        logging.basicConfig(level=logging.INFO)
        logger = logging.getLogger(__name__)
        
        # Construct the API URL
        api_url = f"https://api.github.com/repos/{repo_name}/contents/README.md"
        
        # Setup headers
        headers = {
            "Accept": "application/vnd.github.v3+json"
        }
        if self.gh_token:
            headers["Authorization"] = f"token {self.gh_token}"
        
        try:
            # Make the API request
            response = requests.get(api_url, headers=headers)
            response.raise_for_status()
            
            # Decode the content
            content = response.json()
            if content.get("encoding") == "base64":
                readme_content = base64.b64decode(content["content"]).decode("utf-8")
                return readme_content
            else:
                logger.warning(f"Unexpected content encoding for {repo_name}")
                return None
                
        except requests.exceptions.RequestException as e:
            logger.error(f"Error fetching README for {repo_name}: {str(e)}")
            return None

    def transform(self, example: dict) -> dict:
        repo_name = example['repo_name']
        readme_content = self.fetch_readme_content(repo_name)
        example['readme_exists'] = readme_content is not None
        example['readme'] = readme_content
        return example

    def create_readme_dataset(self, token: Optional[str] = None) -> None:
        """
        Create a DatasetDict containing README contents from multiple repositories.
        
        Args:
            repo_data (List[Dict]): List of dictionaries containing repo_name
            token (str, optional): GitHub personal access token
        
        Returns:
            DatasetDict: Dataset containing README contents
        """
        self.train_set = self.train_set.map(self.transform).filter(lambda x: x['readme_exists'])
        self.test_set = self.test_set.map(self.transform).filter(lambda x: x['readme_exists'])
        self.eval_set = self.eval_set.map(self.transform).filter(lambda x: x['readme_exists'])

Transform dataset

In [41]:
%cd ..
from private_info import gh_access_token
%cd dataset

# IMPORTANT before running this cell check if you have enough API requests left
# This class does not handle the case when you run out of API requests yet
transformed_ds = TransformedDataset(gh_access_token, ds)

/Users/pughal/myRoot/myNUSJourney/Y3S1/CS3210/assi1/solidityGenerator/solGen/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/pughal/myRoot/myNUSJourney/Y3S1/CS3210/assi1/solidityGenerator
/Users/pughal/myRoot/myNUSJourney/Y3S1/CS3210/assi1/solidityGenerator/dataset


Filter: 100%|██████████| 30/30 [00:00<00:00, 2742.03 examples/s]

{'hash': 'ade75f3fcb13d565fd796609e2c90193e25ca33d5ef00b2be67328b8aacfcfec', 'size': 2394, 'ext': '.sol', 'lang': 'Solidity', 'is_test': False, 'repo_id': '519123139', 'repo_name': 'JolyonJian/contracts', 'repo_head': 'b48d691ba0c2bfb014a03e2b15bf7faa40900020', 'repo_path': 'contracts/6087_13421_0xca0840578f57fe71599d29375e16783424023357.sol', 'content_tokens': 320, 'content_chars': 1164, 'content': '// contracts/bridge/L1Helper.sol\n// SPDX-License-Identifier: MIT\n// https://tornado.cash\npragma solidity ^0.7.0;\npragma abicoder v2;\nimport "omnibridge/contracts/helpers/WETHOmnibridgeRouter.sol";\n/// @dev Extension for original WETHOmnibridgeRouter that stores TornadoPool account registrations.\ncontract L1Helper is WETHOmnibridgeRouter {\n  event PublicKey(address indexed owner, bytes key);\n  struct Account {\n    address owner;\n    bytes publicKey;\n  }\n  constructor(IOmnibridge _bridge,\n    IWETH _weth,\n    address _owner) WETHOmnibridgeRouter(_bridge, _weth, _owner) {}\n  f

In [ ]:
print(transformed_ds.train_set.length)

Filter: 100%|██████████| 30/30 [00:00<00:00, 8403.73 examples/s]
